## Этап 1. Изучение Scapy

Изучите основы работы с Scapy по документации:

- структура пакетов;
- перехват трафика;
- фильтрация по протоколам.

Настройте окружение для работы со скриптом Scapy:

Запросом ниже собираем первый дамп трафика при взаимодействии с грюйером


In [ ]:
from scapy.all import conf, sniff, wrpcap
from ipaddress import ip_address
conf.use_pcap = True
conf.ifaces.show()
iface = conf.ifaces.dev_from_index(int(9))
server_ip = ip_address("173.194.222.153")
capture_filter = f"host {server_ip} and (tcp port 443 or udp port 443)"
packets = sniff(iface=iface, filter=capture_filter, timeout=30, started_callback=lambda: print("Capture started", flush=True))
len(packets)
packets[:10].summary()
if packets:
    with open("gruyere-online.pcap", "xb") as output:
        wrpcap(output, packets)

## Этап 2. Анализ трафика

Ознакомьтесь с инструментом Google Gruyere. Google Gruyere работает по протоколу HTTP, поэтому передаваемый трафик не зашифрован и доступен для анализа. Также в документации дана инструкция, как запустить инстанс Google Gruyere локально. Выбирайте более удобный для себя способ.

Запустите Google Gruyere и начните перехват трафика с помощью Scapy.

Взаимодействуйте с сайтом (переходы по страницам, отправка форм, ввод данных), фиксируя трафик.

Проанализируйте перехваченные HTTP-запросы и ответы:

- метод запроса (`GET` / `POST`);
- URL и параметры;
- заголовки;
- тело запроса и ответа.

HTTP-ответы сервера могут быть сжаты (например, `gzip`), необходимо доработать функцию анализа в скрипте для корректного отображения содержимого ответа.

Т.к. в веб версии грюйер шифрует трафик - собрал лог вызовов в файл `.har`

В методе `GET` видим прямую передачу всех параметров запроса в том числе и пароли (помимо всего прочего сохранение происходит `GET` методом):

```text
https://google-gruyere.appspot.com/611941668480270417493193799067911736147/saveprofile?action=update&name=testio&oldpw=&pw=&icon=&web_site=&color=%231254ab&private_snippet=
```

Перехваченный трафик лежит в файлах

- `withoutXSS.har`
- `withoutXSS.pcap`

## Этап 3. Эксплуатация XSS

Проведите рекон-анализ сайта Google Gruyere для поиска потенциальных точек внедрения XSS:

- поля ввода (поиск, комментарии, профиль пользователя);
- параметры в URL (`GET`-запросы);
- данные форм (`POST`-запросы).

Воспользуйтесь документацией Google Gruyere, где подробно описано, как найти и проэксплуатировать XSS уязвимость.

Выполните попытки эксплуатации XSS-уязвимостей, используя найденные точки входа.

Эксплуатацию допускается выполнять:

- через браузер — для визуального подтверждения XSS;
- при этом обязательно вести перехват трафика с помощью Scapy.

В процессе эксплуатации сохраните перехваченный трафик в файл формата `.pcap`.

Для автоматизации тестирования допускается использование параметров скрипта для отправки различных XSS-полезных нагрузок.

Перехваченный трафик лежит в файлах

- `withXSS.har`
- `withXSS.pcap`


## Этап 4. Анализ результатов

Анализ выполнен по файлам `withoutXSS.pcap`, `withXSS.pcap`, `withoutXSS.har` и `withXSS.har` из этой папки. Сетевые пакеты прочитаны через Scapy. Параметры HTTP и HTML-ответы изучены по HAR, поскольку подключение использует HTTPS.

### HTTP-запрос с XSS-полезной нагрузкой

- **В каких параметрах передаётся полезная нагрузка**

  В обновлённом `withXSS.har`, запись № 19, payload передаётся в параметре `snippet` запроса к `/<ID>/newsnippet2`.

- **используется ли GET или POST**:
    GET

- **Как payload выглядит на уровне HTTP**

  Данные находятся в строке запроса после `?`. Тело этих GET-запросов отсутствует. В HAR видны псевдозаголовки HTTP/3 `:method: GET`, `:scheme: https`, `:authority: google-gruyere.appspot.com` и `:path` с путём и параметрами.

  Значение `snippet` после URL-декодирования имеет следующий вид.

  ```html
  <a href="#" onmouseover="alert('You\'ve been XXS\'ed')">&#1055;&#1088;&#1086;&#1074;&#1077;&#1088;&#1082;&#1072; XSS</a>
  ```

### HTTP-ответ сервера

- **Отражается ли XSS-полезная нагрузка в ответе**

  Да. Запрос создания заметки получает ответ `302` с заголовком `Location`, ведущим на `snippets.gtl`. Следующий GET к этой странице получает `200`, и в его HTML сохранён payload. В обновлённом `withXSS.har` это записи № 19 и № 20. Таким образом это сценарий stored XSS.

- **В какой части ответа она присутствует**

  В HTML-теле страницы `snippets.gtl`, внутри `<div id='0'>`, находится элемент `<a>` с атрибутом `onmouseover="alert('You\'ve been XXS\'ed')"`. JavaScript записан в обработчике события HTML-элемента.


### Сравнение трафика

- **Отличия HTTP-ответов при XSS-атаке от обычных запросов**

  В `withoutXSS.har` обычная заметка передаётся как `snippet=Hello`. После перенаправления ответ страницы содержит текст `Hello`. В `withXSS.har` страница возвращает HTML-ссылку с обработчиком `onmouseover`.

  Для обоих вариантов создание заметки даёт `302`, а загрузка страницы заметок — `200`. Основное различие находится в содержимом HTML.

- **Изменения в структуре ответа сервера**

  Вместо обычного текста внутри блока заметки появился элемент `<a>` с JavaScript в атрибуте события.

### Что показывает PCAP

В обновлённом `withXSS.pcap` содержится 51 пакет: 49 UDP и 2 TCP к `64.233.165.153:443`. Захват записан 22.11.2026 в 23:15:11.511–23:15:27.375 МСК. Адрес и время соответствуют обновлённому `withXSS.har`.

Запрос с payload в `snippet` находится в записи HAR № 19. Его интервал — 23:15:14.557–23:15:14.822 МСК. По IP сервера, порту 443 и времени ему соответствуют UDP-пакеты № 21–25 в PCAP. Последующая загрузка `snippets.gtl` находится в записи HAR № 20, её интервал — 23:15:14.822–23:15:15.257 МСК. В этот интервал попадают UDP-пакеты № 26–30. Нумерация пакетов начинается с 1.

UDP-пакеты на порту 443 согласуются с обменом через QUIC. Сопоставление выполнено по сетевым адресам и времени.

### Вывод

Во время проверки в параметрах GET-запросов передавался HTML с обработчиком события. Сервер сохранил заметку и вернул её в HTML страницы с атрибутом `onmouseover`. По сравнению с обычной заметкой в ответе появился фрагмент разметки, который содержит вызов JavaScript. Наличие payload в HTML подтверждено HAR. Визуальное срабатывание `alert` в HAR не фиксируется. Обновлённый `withXSS.pcap` покрывает время исследуемых запросов к тому же IP сервера, что и HAR.
